## Linear Classifier in TensorFlow 
Using Low Level API in Eager Execution mode

### Load tensorflow

In [22]:
#from google.colab import drive
#drive.mount('/content/drive')

In [23]:
#Enable Eager Execution if using tensflow version < 2.0
#From tensorflow v2.0 onwards, Eager Execution will be enabled by default


### Collect Data

In [24]:
import pandas as pd
import tensorflow as tf

In [25]:
tfprice = pd.read_csv('prices.csv')

### Check all columns in the dataset

### Drop columns `date` and  `symbol`

In [26]:
tfprice = tfprice.drop(['date','symbol'], axis =1 )

In [27]:
tfprice.head()

,open,close,low,high,volume
0,123.430000,125.839996,122.309998,126.250000,2163600.0
1,125.239998,119.980003,119.940002,125.540001,2386400.0
2,116.379997,114.949997,114.930000,119.739998,2489500.0
3,115.480003,116.620003,113.500000,117.440002,2006300.0
4,117.010002,114.970001,114.089996,117.330002,1408600.0


### Consider only first 1000 rows in the dataset for building feature set and target set
Target 'Volume' has very high values. Divide 'Volume' by 1000,000

In [28]:
tfprice['volume'] = tfprice['volume']/100000

In [29]:
tfprice = tfprice.head(1000)

### Divide the data into train and test sets

In [30]:
from sklearn.model_selection import train_test_split

X =  tfprice.drop("volume", axis=1)
y =  tfprice.pop("volume")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=1)

#### Convert Training and Test Data to numpy float32 arrays


In [31]:
import numpy as np
X_train = np.asarray(X_train)
y_train = np.asarray(y_train)
X_train.shape

(700, 4)

### Normalize the data
You can use Normalizer from sklearn.preprocessing

In [32]:
from sklearn import preprocessing
X_normalized = preprocessing.normalize(X, norm='l2')

## Building the Model in tensorflow

1.Define Weights and Bias, use tf.zeros to initialize weights and Bias

In [33]:
x = tf.placeholder(shape=[None,4],dtype=tf.float32, name='x-input')
x_n = tf.nn.l2_normalize(x,1)
y_ = tf.placeholder(shape=[None],dtype=tf.float32, name='y-input')

W = tf.Variable(tf.zeros(shape=[4,1]), name="Weights")
b = tf.Variable(tf.zeros(shape=[1]),name="Bias")

2.Define a function to calculate prediction

In [34]:
y = tf.add(tf.matmul(x_n,W),b,name='output')

3.Loss (Cost) Function [Mean square error]

In [35]:
loss = tf.reduce_mean(tf.square(y-y_),name='Loss')

4.Function to train the Model

1.   Record all the mathematical steps to calculate Loss
2.   Calculate Gradients of Loss w.r.t weights and bias
3.   Update Weights and Bias based on gradients and learning rate to minimize loss

In [36]:
train_op = tf.train.GradientDescentOptimizer(0.03).minimize(loss)

## Train the model for 100 epochs 
1. Observe the training loss at every iteration
2. Observe Train loss at every 5th iteration

In [37]:
sess = tf.Session()
sess.run(tf.global_variables_initializer())
training_epochs = 100

In [38]:
for epoch in range(training_epochs):
    _, train_loss = sess.run([train_op,loss],feed_dict={x:X_train, y_:y_train})
    if epoch % 5 == 0:
        print ('Training loss at step: ', epoch, ' is ', train_loss)

Training loss at step:  0  is  22587.957
Training loss at step:  5  is  20389.14
Training loss at step:  10  is  19776.738
Training loss at step:  15  is  19606.19
Training loss at step:  20  is  19558.684
Training loss at step:  25  is  19545.453
Training loss at step:  30  is  19541.77
Training loss at step:  35  is  19540.754
Training loss at step:  40  is  19540.46
Training loss at step:  45  is  19540.377
Training loss at step:  50  is  19540.361
Training loss at step:  55  is  19540.346
Training loss at step:  60  is  19540.357
Training loss at step:  65  is  19540.357
Training loss at step:  70  is  19540.348
Training loss at step:  75  is  19540.352
Training loss at step:  80  is  19540.348
Training loss at step:  85  is  19540.348
Training loss at step:  90  is  19540.35
Training loss at step:  95  is  19540.352


### Get the shapes and values of W and b

In [39]:
W.shape

TensorShape([Dimension(4), Dimension(1)])

In [40]:
b.shape

TensorShape([Dimension(1)])

### Model Prediction on 1st Examples in Test Dataset

In [41]:
test_loss = sess.run([loss],feed_dict={x:X_test, y_:y_test})
print ('Training loss at step:is ', test_loss)

Training loss at step:is  [24101.951]


## Classification using tf.Keras

In this exercise, we will build a Deep Neural Network using tf.Keras. We will use Iris Dataset for this exercise.

### Load the given Iris data using pandas (Iris.csv)

In [42]:
keriris = pd.read_csv('Iris.csv')

In [43]:
keriris.columns

Index(['Id', 'SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm',
       'Species'],
      dtype='object')

### Target set has different categories. So, Label encode them. And convert into one-hot vectors using get_dummies in pandas.

In [44]:
keriris = pd.get_dummies(keriris, prefix=['Species'], columns=['Species'])

In [45]:
keriris.head()

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species_Iris-setosa,Species_Iris-versicolor,Species_Iris-virginica
0,1,5.1,3.5,1.4,0.2,1,0,0
1,2,4.9,3.0,1.4,0.2,1,0,0
2,3,4.7,3.2,1.3,0.2,1,0,0
3,4,4.6,3.1,1.5,0.2,1,0,0
4,5,5.0,3.6,1.4,0.2,1,0,0


In [46]:
keriris.columns

Index(['Id', 'SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm',
       'Species_Iris-setosa', 'Species_Iris-versicolor',
       'Species_Iris-virginica'],
      dtype='object')

In [47]:
type(keriris)

pandas.core.frame.DataFrame

### Splitting the data into feature set and target set

In [48]:
from sklearn.model_selection import train_test_split
X =  keriris.drop(["Species_Iris-setosa","Species_Iris-versicolor","Species_Iris-virginica","Id"], axis=1)
y =  keriris[["Species_Iris-setosa","Species_Iris-versicolor","Species_Iris-virginica"]]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=1)

###  Building Model in tf.keras

Build a Linear Classifier model  <br>
1.  Use Dense Layer  with input shape of 4 (according to the feature set) and number of outputs set to 3<br> 
2. Apply Softmax on Dense Layer outputs <br>
3. Use SGD as Optimizer
4. Use categorical_crossentropy as loss function 

In [49]:
from keras.models import Sequential
from keras.layers import Dense

# create model
model = Sequential()
model.add(Dense(12, input_dim=4, activation='relu'))
model.add(Dense(8, activation='relu'))
model.add(Dense(3, activation='sigmoid'))
# Compile model
model.compile(loss='categorical_crossentropy', optimizer='sgd', metrics=['accuracy'])


Using TensorFlow backend.


### Model Training 

In [50]:
model.fit(X_train,y_train , epochs=100, batch_size=10)

Instructions for updating:
Use tf.cast instead.
Epoch 1/100
105/105 [==============================] - 1s 7ms/step - loss: 1.0447 - acc: 0.5524
Epoch 2/100
105/105 [==============================] - 0s 454us/step - loss: 1.0081 - acc: 0.7238
Epoch 3/100
105/105 [==============================] - 0s 274us/step - loss: 0.9692 - acc: 0.8857
Epoch 4/100
105/105 [==============================] - 0s 195us/step - loss: 0.9142 - acc: 0.7143
Epoch 5/100
105/105 [==============================] - 0s 201us/step - loss: 0.8613 - acc: 0.6952
Epoch 6/100
105/105 [==============================] - 0s 224us/step - loss: 0.8185 - acc: 0.6952
Epoch 7/100
105/105 [==============================] - 0s 206us/step - loss: 0.7847 - acc: 0.6952
Epoch 8/100
105/105 [==============================] - 0s 291us/step - loss: 0.7587 - acc: 0.6952
Epoch 9/100
105/105 [==============================] - 0s 262us/step - loss: 0.7329 - acc: 0.6952
Epoch 10/100
105/105 [==============================] - 0s 238us/step - 

105/105 [==============================] - 0s 136us/step - loss: 0.1922 - acc: 0.9524
Epoch 81/100
105/105 [==============================] - 0s 173us/step - loss: 0.1688 - acc: 0.9524
Epoch 82/100
105/105 [==============================] - 0s 148us/step - loss: 0.1801 - acc: 0.9429
Epoch 83/100
105/105 [==============================] - 0s 188us/step - loss: 0.1767 - acc: 0.9429
Epoch 84/100
105/105 [==============================] - 0s 185us/step - loss: 0.2749 - acc: 0.8667
Epoch 85/100
105/105 [==============================] - 0s 238us/step - loss: 0.1712 - acc: 0.9333
Epoch 86/100
105/105 [==============================] - 0s 175us/step - loss: 0.1632 - acc: 0.9714
Epoch 87/100
105/105 [==============================] - 0s 190us/step - loss: 0.2078 - acc: 0.9048
Epoch 88/100
105/105 [==============================] - 0s 230us/step - loss: 0.1871 - acc: 0.9429
Epoch 89/100
105/105 [==============================] - 0s 175us/step - loss: 0.1770 - acc: 0.9238
Epoch 90/100
105/105 [=

### Model Prediction

In [52]:
iris_scores = model.evaluate(X_train, y_train)
iris_scores[1]*100

105/105 [==============================] - 0s 55us/step


89.52380963734218

### Save the Model

In [54]:
model.save('iris.jason')

### Build and Train a Deep Neural network with 2 hidden layer  - Optional - For Practice

Does it perform better than Linear Classifier? What could be the reason for difference in performance?